In [1]:
import torch
import matplotlib.pyplot as plt

In [2]:
#choose cpu/gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
#ODE sampler
with torch.no_grad():
    def ODE_sampler(model, steps=50):
        #random generate img
        x = torch.randn(64, 3, 32, 32).to(device)
        #linear generate sigma
        sigmas = torch.linspace(80, 0.002, steps).to(device)
        #generate img without noise
        for sigma in sigmas:
            #reshape sigma
            sigma = sigma.view(1, 1, 1, 1)
            #calculate predictive value
            pred = model(x, sigma)
            #calculate original img
            d = (x - pred) / sigma
            x = x - d * (sigmas[0] - sigmas[1])
        return x

In [6]:
#show sampler
def show_sampler(samples):
    #choose cpu
    samples = samples.detach().cpu()
    #reverse normalization
    samples = (samples + 1) / 2
    samples = torch.clamp(samples, 0, 1)

    grid = samples[:3]
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    for i, ax in enumerate(axes.flatten()):
        ax.imshow(grid[i].permute(1, 2, 0))
        ax.axis('off')
    plt.show()